# Lesson 25 · RNNs are back (linear attention & Mamba): compress the past into a fixed-size matrix, then measure how much it remembers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/25-rnn-tro-lai-en.ipynb)

This notebook accompanies [Lesson 25 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rnn-tro-lai/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/25-rnn-tro-lai.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
Attention keeps the keys and values of every token it has read, so its memory grows with the length of the text.
Drop the softmax, and the same computation can be rewritten as a recurrent network with a **fixed-size state** $S$
of size $d_k \times d_v$. The central question: how much can such a matrix remember, and how do modern layers (forget
gates, the delta rule, Mamba-2) manage it?

**What you'll do:**
1. Compute Llama's KV cache from its published configuration, and see it swell with length.
2. Write linear attention in three ways (recurrent, masked matrix, chunked) and check that they agree.
3. Write $N$ key–value pairs into an untrained $16 \times 16$ matrix, measure how many can be read back, then compare
   with 7 trained layers.
4. See that the delta rule is one step of online least-squares regression, and train a layer whose gate learns by
   itself to forget.
5. Check that Mamba-2 is exactly gated linear attention, then compute the memory of hybrid models at 262,144 tokens.

**Background:** [Lesson 15 · RNN & LSTM](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rnn/) (hidden state, forget
gate), [Lesson 16 · The attention mechanism](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/attention/) (query, key,
value) and [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/) (causal
mask, KV cache). **Run time:** under a minute on a CPU, including the first download of two results files (~0.1 MB).

Every number in the notebook is computed by the cells below (or read from the results files the demo uses), and
matches the numbers in the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `fmt()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign. `sci()` prints very small or very
large numbers in the form $4.6 \cdot 10^{-14}$.

In [ ]:
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
SEQUENTIAL = LinearSegmentedColormap.from_list("white_blue", ["#ffffff", BLUE])  # 0 → 1
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def fmt(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never prints “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return fmt(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """A number in scientific notation, in the English format: 4.6·10⁻¹⁴."""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)))
    m = x / 10.0**e
    if round(abs(m), digits) >= 10:
        m, e = m / 10, e + 1
    return f"{fmt(m, digits)}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return fmt(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it is
    long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(fmt(262144, 0), fmt(0.9995, 4), pct(0.7902), sci(4.64e-14))

## 2 · Data: the demo's two results files

This lesson needs no external dataset: the key–value sequences are generated at random right in the notebook. But two
things take too long to redo here, so we load the very files the web demo uses (with an MD5 check):
- `linear-recall.json`: 74 token-mixer configurations × 3 seeds, each run 6,000 training steps on the key–value recall
  task (about 400 CPU minutes), along with the gates the seed-0 models learned and the table of the untrained memory;
- `linear-cost.json`: the number of layers, number of heads and state size of 10 real models, read from their
  published `config.json`.

In each section, the notebook runs a small version of the mechanism live before reading the full results.

In [ ]:
# The results of a long training run (the same file as the demo on the web). build.py replaces COURSE_DATA_URL with
# the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


REC = fetch_json("linear-recall.json", "978ff84d1924cb1d79e398eaaea8e026")  # the demo's training results
COST = fetch_json("linear-cost.json", "fa39c9c60a611e5ebf6f432ae4878a7f")  # the demo's model configurations
print(
    f"linear-recall.json: {len(REC['runs'])} configurations × {len(REC['setup']['seeds'])} seeds, "
    f"{fmt(REC['setup']['steps'], 0)} steps each; {fmt(REC['setup']['nVal'], 0)} validation sequences, "
    f"{fmt(REC['setup']['nTest'], 0)} test sequences"
)
print("linear-cost.json:", ", ".join(m["name"] for m in COST["models"]))
size = sum((Path("data") / n).stat().st_size for n in ("linear-recall.json", "linear-cost.json"))
print(f"{fmt(size / 1e6, 1)} MB in total")

## 3 · The KV cache swells with length

When generating text, each attention layer keeps the key $k$ and the value $v$ of every token it has read (Lesson
17), so that a new token doesn't have to recompute them. For each token, each layer adds $2 \cdot n_{kv} \cdot d_{head}$
numbers to the cache: the K and V of each KV head. We take the published configurations of two 32-layer models, with
each number stored in bf16 (2 bytes):
- **Llama 2 7B** (Touvron et al., 2023): 32 query heads, 32 KV heads, 128 dimensions per head;
- **Llama 3 8B** (Grattafiori et al., 2024): 32 query heads but only 8 KV heads (grouped-query attention, GQA:
  4 query heads share one pair of K, V).

### 🤔 Predict first
Llama 3 8B is a little larger than Llama 2 7B. At 262,144 tokens, which model's KV cache is larger, and by what factor?

In [ ]:
KiB, MiB, GiB = 2**10, 2**20, 2**30
T_LONG = 262_144  # 2¹⁸ tokens ("256K"), the native context of Jamba and Qwen3-Next

LLAMA = {  # published config.json; Llama 2 uses GQA only in its 34B and 70B versions
    "Llama 3 8B": {"layers": 32, "heads": 32, "kv_heads": 8, "head_dim": 128},
    "Llama 2 7B": {"layers": 32, "heads": 32, "kv_heads": 32, "head_dim": 128},
}


def kv_per_token(attn_layers, kv_heads, head_dim, bytes_per_number=2):
    """Bytes each token adds to the cache: the K and V of every KV head, in every attention layer (bf16 = 2 bytes per
    number)."""
    return attn_layers * 2 * kv_heads * head_dim * bytes_per_number


for name, c in LLAMA.items():
    per = kv_per_token(c["layers"], c["kv_heads"], c["head_dim"])
    sizes = " · ".join(f"{fmt(T, 0)} tokens: {fmt(per * T / GiB, 1)} GiB" for T in (4096, 32768, T_LONG, 4 * T_LONG))
    print(f"{name}: {fmt(per / KiB, 0)} KiB per token  →  {sizes}")
kv3, kv2 = (kv_per_token(c["layers"], c["kv_heads"], c["head_dim"]) for c in LLAMA.values())
assert kv3 == 128 * KiB and kv3 * T_LONG == 32 * GiB, "differs from the lesson: check the configuration again"
assert kv2 * T_LONG == 128 * GiB  # exactly LLaMA-2's 128GB in Table 1 of the Jamba paper
reads = T_LONG * (T_LONG + 1) // 2
print(
    f"generating {fmt(T_LONG, 0)} tokens: token t reads back the t tokens stored → {sci(reads)} reads in total "
    "(grows as T²)"
)

Llama 2 7B needs 4 times as much: 128 GiB, against 32 GiB for Llama 3 8B, even though it has fewer parameters. The
reason is GQA: Llama 3 keeps only 8 pairs of K, V per layer instead of 32. The 128 GiB for Llama 2 matches Table 1 of
the Jamba paper (Lieber et al., 2024), which lists it as “128GB” at 256K tokens and 16 bits.

And memory is only half the problem. Each new token has to read through the *whole* cache again: token $t$ reads $t$
pairs $(k, v)$, so generating $T$ tokens takes about $T^2/2$ reads in total. GQA only shrinks the constant; the memory
still grows linearly with $T$.

## 4 · Drop the softmax: the same computation, written two ways

Attention with a causal mask (Lesson 17), in the row convention ($q_t, k_t$ are $1 \times d_k$ row vectors, $v_t$ is
$1 \times d_v$):

$$o_t = \frac{\sum_{s \le t} \exp(q_t k_s^\top/\sqrt{d_k})\, v_s}{\sum_{s \le t} \exp(q_t k_s^\top/\sqrt{d_k})}$$

Katharopoulos et al. (2020) replace $\exp(q k^\top)$ with $\varphi(q)\varphi(k)^\top$, where $\varphi(x) = \operatorname{elu}(x) + 1$
(always positive). Then $\varphi(q_t)$ can be pulled out of the sum, by the associativity of matrix multiplication:

$$o_t = \frac{\varphi(q_t) \sum_{s \le t} \varphi(k_s)^\top v_s}{\varphi(q_t) \sum_{s \le t} \varphi(k_s)^\top}
= \frac{\varphi(q_t)\, S_t}{\varphi(q_t)\, z_t^\top}, \qquad S_t = S_{t-1} + \varphi(k_t)^\top v_t, \quad z_t = z_{t-1} + \varphi(k_t)$$

Written attention-style, we compute the masked $T \times T$ matrix $\big(\varphi(Q)\varphi(K)^\top \odot M\big)V$ and
divide by the sum of each row. Written RNN-style, we carry only a **state** $S_t$ of size $d_k \times d_v$ (and $z_t$ of
size $1 \times d_k$), updated once per token. The cell below computes both, and softmax attention too for comparison,
on a random sequence.

In [ ]:
def elu1(x):
    """φ(x) = elu(x) + 1: always positive, so the sum in the denominator is never 0."""
    return np.where(x > 0, x + 1, np.exp(np.minimum(x, 0)))


def softmax_attention(Q, K, V):
    s = Q @ K.T / np.sqrt(Q.shape[1])
    s = np.where(np.tri(len(Q), dtype=bool), s, -np.inf)  # causal mask: token t looks only at s ≤ t
    P = np.exp(s - s.max(1, keepdims=True))
    return (P / P.sum(1, keepdims=True)) @ V


def linear_attention_matrix(Q, K, V):
    A = (elu1(Q) @ elu1(K).T) * np.tri(len(Q))  # a masked T × T matrix, no softmax
    return (A @ V) / A.sum(1, keepdims=True)


def linear_attention_rnn(Q, K, V):
    S, z = np.zeros((K.shape[1], V.shape[1])), np.zeros(K.shape[1])  # fixed-size state: d_k × d_v and 1 × d_k
    O = np.zeros_like(V)
    for t in range(len(Q)):
        S += np.outer(elu1(K[t]), V[t])  # S_t = S_{t−1} + φ(k_t)ᵀ v_t
        z += elu1(K[t])
        O[t] = elu1(Q[t]) @ S / (elu1(Q[t]) @ z)
    return O


rng = np.random.default_rng(0)
T, d = 64, 16
Q, K, V = rng.standard_normal((3, T, d))
O_mat, O_rnn, O_soft = linear_attention_matrix(Q, K, V), linear_attention_rnn(Q, K, V), softmax_attention(Q, K, V)
print(f"the matrix form and the RNN form differ by at most {sci(np.abs(O_mat - O_rnn).max())}")
print(f"linear attention and softmax attention differ by {fmt(np.abs(O_mat - O_soft).mean(), 3)} on average")
print(
    f"memory at the last token: softmax keeps {fmt(T * 2 * d, 0)} numbers (T·2d), the RNN form keeps "
    f"{fmt(d * d + d, 0)} numbers (d² + d)"
)
assert np.abs(O_mat - O_rnn).max() < 1e-10

The two ways agree to rounding error, but they differ clearly from softmax attention: this is a *different* model,
not a faster way of computing the same model. From token $d/2 + 1 = 9$ on, softmax's $T \cdot 2d$ cache is already
larger than the $d \times d$ state; at 64 tokens it is almost 8 times as large, and it keeps growing.

### The general form: a forget gate $\alpha_t$ and a write strength $\beta_t$

Today's layers drop $\varphi$ and the denominator (the delta family also normalizes $q, k$ to length 1; here we do that
for every layer), and add two numbers for each token:

$$S_t = \alpha_t\, S_{t-1} + \beta_t\, k_t^\top v_t, \qquad o_t = q_t S_t$$

- $\alpha_t \in (0, 1)$ is the **forget gate**: it shrinks the whole state before writing (like the LSTM's $f_t$ in
  Lesson 15, but only a single number per head);
- $\beta_t$ is the **write strength**.

Substituting $S_{t-1}$ back in gives the attention form: $o_t = \sum_{s \le t} \big(\prod_{r=s+1}^{t} \alpha_r\big)\, \beta_s\,
(q_t k_s^\top)\, v_s$. As matrices: $O = (QK^\top \odot M)\operatorname{diag}(\beta)V$, where the mask $M_{ts} =
\prod_{r=s+1}^{t} \alpha_r$ for $s \le t$ (and 0 if $s > t$). With $\alpha = 1$, $M$ is exactly the causal mask of Lesson 17.

There is a third way, the one GPUs use during training: **splitting into chunks** of $C$ tokens. Within each chunk, we
compute it like masked attention (matrix multiplications); between chunks, we pass only the state $S$. The RNN of
Lesson 15 can't do this because the $\tanh$ sits inside the loop; here $S_t$ depends linearly on $S_{t-1}$, so the
steps can be separated.

The cell below generates sequences identical to the demo's (JavaScript's `mulberry32` random number generator, ported
to numpy, and the Box–Muller transform), so that the results match what the browser computes. The function `sequence`
has four kinds of gate: no gate, a fixed $\alpha$, a per-token $\alpha$ (now and then close to 0, like an “erase”), and
a long kind for the 2,048-token test.

In [ ]:
def mulberry32(seed, n):
    """The first n numbers of JavaScript's mulberry32(seed) generator (the demo uses exactly this generator)."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def box_muller(u):
    """Each pair of uniform numbers (u₁, u₂) gives one standard normal number N(0, 1)."""
    return np.sqrt(-2 * np.log(1 - u[..., 0::2])) * np.cos(2 * np.pi * u[..., 1::2])


def unit(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)


def sequence(seed, T, dk, dv, gate, alpha=0.9):
    """Each token, in order: q (d_k standard normal numbers), k (d_k), v (d_v), then two uniform numbers g₁, g₂ for the
    gate."""
    u = mulberry32(seed, T * (4 * dk + 2 * dv + 2)).reshape(T, -1)
    Q, K = unit(box_muller(u[:, : 2 * dk])), unit(box_muller(u[:, 2 * dk : 4 * dk]))
    V = box_muller(u[:, 4 * dk : 4 * dk + 2 * dv])
    g1, g2 = u[:, -2], u[:, -1]
    a = {
        "none": np.ones(T),
        "fixed": np.full(T, alpha),
        "token": np.where(g1 < 0.08, 0.05 + 0.1 * g2, 0.9 + 0.1 * g2),  # 8% “erase” tokens (α < 0.15)
        "long": 0.9 + 0.1 * g1,
    }[gate]
    b = 0.5 + 0.5 * g2 if gate == "long" else np.ones(T)
    return Q, K, V, a, b


Q, K, V, a, b = sequence(7, 64, 4, 4, "token")
print(
    "q₁ =",
    ", ".join(fmt(x, 3) for x in Q[0]),
    " (length 1)  ·  α of the first 12 tokens:",
    " ".join(fmt(x, 2) for x in a[:12]),
)

The three functions below compute the same $O$ in three ways. The function `decay` builds the mask $M$ from the
cumulative sum $c_t = \sum_{u \le t} \log\alpha_u$, because $\prod_{r=s+1}^{t} \alpha_r = e^{c_t - c_s}$. In the chunked
form, token $t$ of a chunk reads the state carried over from the previous chunk (shrunk further by $e^{c_t}$) plus the
masked attention part within the chunk; at the end of the chunk, the state is updated once, with one matrix
multiplication.

In [ ]:
def recurrent(Q, K, V, a, b):
    """RNN form: one d_k × d_v state, updated token by token."""
    S, O = np.zeros((K.shape[1], V.shape[1])), np.zeros_like(V)
    for t in range(len(a)):
        S = a[t] * S + b[t] * np.outer(K[t], V[t])  # S_t = α_t S_{t−1} + β_t k_tᵀ v_t
        O[t] = Q[t] @ S  # o_t = q_t S_t
    return O


def decay(a):
    """c_t = Σ_{u≤t} log α_u and the mask M_ts = ∏_{r=s+1..t} α_r = exp(c_t − c_s) when s ≤ t, 0 when s > t."""
    c = np.cumsum(np.log(a))
    return c, np.exp(np.where(np.tri(len(a), dtype=bool), c[:, None] - c[None, :], -np.inf))


def parallel(Q, K, V, a, b):
    """Attention form: a masked T × T matrix, O = (QKᵀ ⊙ M) diag(β) V."""
    _, M = decay(a)
    return ((Q @ K.T) * M * b[None, :]) @ V


def chunked(Q, K, V, a, b, C):
    """Chunked form with C tokens per chunk: masked within the chunk, the state passed between chunks."""
    S, O = np.zeros((K.shape[1], V.shape[1])), np.zeros_like(V)
    for t0 in range(0, len(a), C):
        blk = slice(t0, min(len(a), t0 + C))
        c, M = decay(a[blk])
        O[blk] = np.exp(c)[:, None] * (Q[blk] @ S) + ((Q[blk] @ K[blk].T) * M * b[None, blk]) @ V[blk]
        S = np.exp(c[-1]) * S + (K[blk] * (np.exp(c[-1] - c) * b[blk])[:, None]).T @ V[blk]
    return O


# (seed, T, d_k, d_v, gate, α, chunk C): the demo's sequences (d 4, T ≤ 64) and the 2,048-token test (d 16, C 64)
CASES = [
    (7, 64, 4, 4, "none", 0.9, 8),
    (7, 64, 4, 4, "fixed", 0.8, 8),
    (7, 64, 4, 4, "token", 0.9, 16),
    (3, 37, 4, 4, "token", 0.9, 4),
    (1, 2048, 16, 16, "long", 0.9, 64),
]
for seed, T, dk, dv, gate, alpha, C in CASES:
    s = sequence(seed, T, dk, dv, gate, alpha)
    O = recurrent(*s)
    e_par, e_chk = np.abs(O - parallel(*s)).max(), np.abs(O - chunked(*s, C)).max()
    print(
        f"T = {T:>5} · d = {dk:>2} · gate {gate:<5}: |o| up to {fmt(np.abs(O).max(), 2):>5}; "
        f"matrix off by {sci(e_par)}, chunked (C = {C}) off by {sci(e_chk)}"
    )
    assert max(e_par, e_chk) < 1e-10, "the three forms must agree: check the code again"

The three ways give the same result for every kind of gate. The largest deviation, at 2,048 tokens, is only of order
$10^{-14}$ (the matrix form adds 2,048 terms at once, the recurrent form accumulates them in a different order), and
the chunked form deviates by about $10^{-15}$. That is the rounding error of 64-bit floating-point numbers, not an
error of the method.

The figure below shows the sequence with a per-token gate ($T = 64$). On the left is the decay mask $M_{ts}$: each row
$t$ tells how much of an old token $s$ is left when token $t$ reads the state. The “erase” tokens ($\alpha_t < 0.15$,
gray ticks on the axis) cut the mask into nearly independent blocks. On the right are the actual weights
$(q_t \cdot k_s)\,M_{ts}\,\beta_s$ of the matrix form: like the attention maps of Lesson 16, but without a softmax, so
there are both negative (orange) and positive (blue) weights, and the rows don't sum to 1.

In [ ]:
Q, K, V, a, b = sequence(7, 64, 4, 4, "token")
_, M = decay(a)
W = (Q @ K.T) * M * b[None, :]
resets = np.flatnonzero(a < 0.15)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
im0 = axes[0].imshow(M, cmap=SEQUENTIAL, vmin=0, vmax=1)
vmax = np.abs(W).max()
im1 = axes[1].imshow(W, cmap=DIVERGING, vmin=-vmax, vmax=vmax)
for ax, im, title in ((axes[0], im0, "decay mask M (product of the gates α)"), (axes[1], im1, "weight (qₜ·kₛ)·Mₜₛ·βₛ")):
    ax.set(xlabel="token being read, s", ylabel="reading token, t", title=title)
    ax.plot(resets, np.full(len(resets), 63.5), "|", color=GREY, ms=14, mew=2, clip_on=False)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
fmt_axes(fig)
plt.show()
print(
    f"{len(resets)} “erase” tokens at positions {', '.join(map(str, resets))}; their α: {', '.join(fmt(x, 3) for x in a[resets])}"
)

### The delta rule in three forms

Linear attention only *adds* to the state. The delta rule (Schlag, Irie & Schmidhuber, 2021) writes the
**difference** between the new value and the value the state currently holds for that key:

$$u_t = v_t - \alpha_t\, k_t S_{t-1}, \qquad S_t = \alpha_t S_{t-1} + \beta_t\, k_t^\top u_t
\quad\Longleftrightarrow\quad S_t = \alpha_t \big(I - \beta_t k_t^\top k_t\big) S_{t-1} + \beta_t\, k_t^\top v_t$$

With $\alpha = 1$ this is DeltaNet; with a per-token $\alpha_t$ it is Gated DeltaNet (Yang, Kautz & Hatamizadeh, 2024).
The update is still linear in $S$, so it still has a matrix form; we only need to replace $V$ with $U$: substituting
$S_{t-1}$ into $u_t$ gives $u_t + \sum_{s<t} M_{ts}\,\beta_s\,(k_t \cdot k_s)\, u_s = v_t$, i.e. a lower-triangular
system $(I + L)U = V$ with $L_{ts} = M_{ts}\beta_s (k_t \cdot k_s)$ for $s < t$ (the “UT” form that DeltaNet uses for
chunkwise training).

In [ ]:
def recurrent_delta(Q, K, V, a, b):
    S, O = np.zeros((K.shape[1], V.shape[1])), np.zeros_like(V)
    for t in range(len(a)):
        u = V[t] - a[t] * (K[t] @ S)  # the difference: v_t minus the value stored for k_t
        S = a[t] * S + b[t] * np.outer(K[t], u)
        O[t] = Q[t] @ S
    return O


def solve_u(K, V, b, M, R=0):
    """(I + L) U = V − R, with L_ts = M_ts β_s (k_t·k_s) when s < t (lower triangular, ones on the diagonal)."""
    L = np.tril(M * b[None, :] * (K @ K.T), -1)
    return np.linalg.solve(np.eye(len(b)) + L, V - R)


def parallel_delta(Q, K, V, a, b):
    _, M = decay(a)
    return ((Q @ K.T) * M * b[None, :]) @ solve_u(K, V, b, M)


def chunked_delta(Q, K, V, a, b, C):
    S, O = np.zeros((K.shape[1], V.shape[1])), np.zeros_like(V)
    for t0 in range(0, len(a), C):
        blk = slice(t0, min(len(a), t0 + C))
        c, M = decay(a[blk])
        e = np.exp(c)
        U = solve_u(K[blk], V[blk], b[blk], M, e[:, None] * (K[blk] @ S))  # minus the previous chunk's carried state
        O[blk] = e[:, None] * (Q[blk] @ S) + ((Q[blk] @ K[blk].T) * M * b[None, blk]) @ U
        S = np.exp(c[-1]) * S + (K[blk] * (np.exp(c[-1] - c) * b[blk])[:, None]).T @ U
    return O


for seed, T, dk, dv, gate, alpha, C in CASES:
    s = sequence(seed, T, dk, dv, gate, alpha)
    O = recurrent_delta(*s)
    e_par, e_chk = np.abs(O - parallel_delta(*s)).max(), np.abs(O - chunked_delta(*s, C)).max()
    print(f"delta, T = {T:>5} · gate {gate:<5}: matrix off by {sci(e_par)}, chunked (C = {C}) off by {sci(e_chk)}")
    assert max(e_par, e_chk) < 1e-10, "the three forms must agree: check the code again"

The delta rule also gives three forms that agree to rounding error. Its price is solving a $C \times C$ triangular
system in each chunk, instead of only multiplying matrices.

## 5 · How many key–value pairs can a 16 × 16 matrix remember?

The state doesn't grow, so at some point it is bound to fill up. To measure that, we use an **untrained** memory,
exactly like the demo's second card:
- a table of 64 value “words”, each a random unit vector in $\mathbb{R}^{16}$;
- $N$ pairs: each pair has a random unit key $k_i$ ($1 \times 16$) and a value word $v_i$ chosen at random;
- write all $N$ pairs into $S$ ($16 \times 16$), then query each key again: $o_i = k_i S$, and guess the word with the
  largest dot product with $o_i$. It is correct if that is $v_i$.

We average over 400 random draws, with the same random numbers the demo uses.

Four ways of writing:
- **add:** $S \leftarrow S + k_i^\top v_i$ (linear attention);
- **gated:** $S \leftarrow 0.9\, S + k_i^\top v_i$ (a fixed forget gate $\alpha = 0.9$);
- **delta:** $S \leftarrow S + k_i^\top (v_i - k_i S)$ ($\beta = 1$);
- **least squares:** the matrix $S$ that minimizes $\sum_i \lVert k_i S - v_i \rVert^2$, computed from all the pairs at
  once: the matrix that fits the values best in squared error.

In [ ]:
CODEWORDS = 64  # number of value words


def memory_draws(draws=400, dk=16, nmax=64):
    """As in the demo: draw d uses mulberry32(1 + d); first the 64 value words, then each pair (a key of d_k numbers,
    then the word's index)."""
    per = 2 * dk + 1
    U = np.stack([mulberry32(1 + d, 2 * CODEWORDS * dk + nmax * per) for d in range(draws)])
    code = unit(box_muller(U[:, : 2 * CODEWORDS * dk]).reshape(draws, CODEWORDS, dk))
    pairs = U[:, 2 * CODEWORDS * dk :].reshape(draws, nmax, per)
    keys = unit(box_muller(pairs[..., : 2 * dk]))  # (draw, pair, d_k), length 1
    ids = np.floor(pairs[..., 2 * dk] * CODEWORDS).astype(int)
    vals = np.take_along_axis(code, ids[..., None], 1)  # v_i is word number ids[i]
    return code, keys, ids, vals


def read_back(S, code, K, ids):
    """Query every key: o = kS, guess the word closest to o (largest dot product). Returns an array of right/wrong."""
    O = K @ S
    return (O @ code.transpose(0, 2, 1)).argmax(-1) == ids


def write(rule, K, V, alpha=0.9):
    """Write the pairs (rows of K, V) into S, for every draw at once."""
    S = np.zeros((len(K), K.shape[2], V.shape[2]))
    if rule == "ls":
        Kt = K.transpose(0, 2, 1)
        if K.shape[1] <= K.shape[2]:  # fewer pairs than dimensions: fit every pair exactly, S = Kᵀ(KKᵀ)⁻¹V
            return Kt @ np.linalg.solve(K @ Kt, V)
        return np.linalg.solve(Kt @ K, Kt @ V)  # more pairs: S = (KᵀK)⁻¹KᵀV
    for i in range(K.shape[1]):
        k, v = K[:, i], V[:, i]
        if rule == "delta":
            v = v - np.einsum("bi,bij->bj", k, S)  # write only the difference
        S = (alpha if rule == "gate" else 1.0) * S + k[:, :, None] * v[:, None, :]
    return S


code, keys, ids, vals = memory_draws()
N = 4
S = write("add", keys[:1, :N], vals[:1, :N])
ok = read_back(S, code[:1], keys[:1, :N], ids[:1, :N])[0]
print(
    f"draw 0, {N} pairs: S is {S.shape[1]} × {S.shape[2]}; values to read {ids[0, :N].tolist()}, read correctly {ok.sum()}/{N}"
)
print(
    f"dot products between the keys: {' '.join(fmt(x, 2) for x in (keys[0, :N] @ keys[0, :N].T)[np.triu_indices(N, 1)])}"
)

Reading a key back gives $o_i = k_i S = \sum_j (k_i \cdot k_j)\, v_j = v_i + \sum_{j \ne i} (k_i \cdot k_j)\, v_j$: the
correct value plus **overlap noise** from every other pair. Random keys in 16 dimensions are not orthogonal to one
another (dot products of about $\pm 0.25$), so the more pairs, the larger the noise.

### 🤔 Predict first
A $16 \times 16$ matrix has exactly 16 independent directions. Write **16 pairs** with random keys by adding: about
what percentage is read back correctly? And the delta rule, which “corrects” the value before writing: is it right
more often or less often than adding?

In [ ]:
RULES = {"add": "add kᵀv", "gate": "gated, α = 0.9", "delta": "delta rule", "ls": "least squares"}
NS = np.arange(1, 65)


def capacity(rule, keys, vals, code, ids):
    """Recall accuracy when writing N = 1…64 pairs (averaged over all draws), and the right/wrong array for each N."""
    acc, hits = [], []
    for N in NS:
        S = write(rule, keys[:, :N], vals[:, :N])
        h = read_back(S, code, keys[:, :N], ids[:, :N])
        acc.append(h.mean())
        hits.append(h)
    return np.array(acc), hits


table, hits = {}, {}
for rule in RULES:
    table[rule], hits[rule] = capacity(rule, keys, vals, code, ids)
for N in (8, 16, 32, 64):
    print(f"N = {N:>2}: " + " · ".join(f"{RULES[r]} {pct(table[r][N - 1], 1)}" for r in RULES))
stored = REC["untrained"]["rand"]  # the table the demo computes in JavaScript with the same draws
worst = max(np.abs(table[r] - np.array(stored[r]["acc"])).max() for r in RULES)
print(f"against the demo's table (rounded to 4 digits): off by at most {fmt(100 * worst, 3)} percentage points")
assert worst < 1e-3, "differs from the demo: check the random draws again"

The figure below shows recall accuracy against the number of pairs $N$ for the four ways of writing; each has its own
color and marker. The vertical dashed line marks $N = d_k = 16$.

In [ ]:
STYLE = {"add": (BLUE, "o", "-"), "gate": (ORANGE, "^", "--"), "delta": (GREEN, "D", "-"), "ls": ("black", "s", ":")}
fig, ax = plt.subplots(figsize=(8, 3.8))
for rule, label in RULES.items():
    color, marker, ls = STYLE[rule]
    ax.plot(NS, 100 * table[rule], color=color, marker=marker, ls=ls, ms=4, markevery=4, lw=1.8, label=label)
ax.axvline(16, color=GREY, lw=1, ls="--")
ax.text(16.8, 6, "N = d_k = 16", color=GREY, fontsize=9)
ax.set(xlabel="number of pairs written, N", ylabel="read back correctly (%)", xlim=(1, 64), ylim=(0, 102))
ax.set_title("Untrained 16 × 16 memory, random keys (average of 400 draws)")
ax.legend(frameon=False, loc="upper right")
ax.grid(alpha=0.3)
fmt_axes(fig)
plt.show()

Adding is only 83.1% correct at 16 pairs, even though the matrix has all 16 directions: random keys overlap, and the
overlap noise adds up. If the keys are **orthogonal**, $k_i \cdot k_j = 0$ for $i \ne j$, then $o_i = v_i$ exactly, for
up to $d_k$ pairs. The cell below orthogonalizes the keys (Gram–Schmidt, via `np.linalg.qr`) and writes them again.

In [ ]:
N = 16
Qr = np.linalg.qr(keys[:, :N].transpose(0, 2, 1))[0]  # orthonormal columns, same subspace as the first N keys
K_orth = Qr.transpose(0, 2, 1)
for rule in ("add", "delta"):
    acc_orth = read_back(write(rule, K_orth, vals[:, :N]), code, K_orth, ids[:, :N]).mean()
    print(
        f"{N} orthogonal keys, {RULES[rule]}: {pct(acc_orth, 1)} correct  (random keys: {pct(table[rule][N - 1], 1)})"
    )
    assert acc_orth == 1.0
q_gate = [h.mean() for h in np.array_split(hits["gate"][31], 4, axis=1)]
q_add = [h.mean() for h in np.array_split(hits["add"][31], 4, axis=1)]
q_delta = [h.mean() for h in np.array_split(hits["delta"][31], 4, axis=1)]
print("32 pairs, split into 4 groups in writing order (old → new):")
for name, q in (("add", q_add), ("gated, α = 0.9", q_gate), ("delta", q_delta)):
    print(f"  {name:<16} " + " / ".join(pct(x, 0) for x in q))
print(f"least squares: {pct(table['ls'][31], 1)} at 32 pairs, {pct(table['ls'][63], 1)} at 64 pairs")

Three different kinds of failure:
- **Adding** errs evenly in every group (about 53–55% per group with 32 pairs): that is **overlap**, not forgetting.
- **A fixed gate** $\alpha = 0.9$ leaves a pair written $n$ steps ago with only $0.9^n$ of its magnitude. The newest
  group is 89% correct, the oldest 3%: this is real **forgetting**, and it isn't selective.
- **The untrained delta rule** does even *worse* than adding. At each write, it corrects $S$ so that the new key reads
  out exactly the new value; the part of $S$ it corrects is the direction of $k_t$, and that direction overlaps the old
  keys, so the old values gradually get overwritten. Its errors, too, lean toward the old pairs.

The least-squares solution fits every pair exactly when $N \le 16$, but at 32 pairs it is only 79.0% correct. The
system $KS = V$ has $N$ rows, while each column of $S$ has only $d_k = 16$ unknowns, so with $N > d_k$ arbitrary keys no
matrix returns every value *exactly*. Memory grows with $d_k$, not with the length of the text. (Reading correctly only
requires the value read out to be closer to the right word than to every other word, a looser condition; section 7
comes back to this.)

## 6 · The delta rule is online regression

Why is it called “delta”? Treat $S$ as a linear model that predicts a value from a key, $\hat v = kS$, and measure the
error on just one pair:

$$\mathcal{L}_t(S) = \tfrac12 \lVert k_t S - v_t \rVert^2, \qquad \nabla_S \mathcal{L}_t = k_t^\top (k_t S - v_t)$$

One step of gradient descent (Lesson 05) with learning rate $\eta = \beta_t$ gives $S - \beta_t k_t^\top (k_t S - v_t) =
S + \beta_t k_t^\top (v_t - k_t S)$: exactly the delta rule. Linear attention, by contrast, is one step on the *linear*
loss $-\langle k_t S, v_t\rangle$, whose gradient $-k_t^\top v_t$ doesn't depend on $S$: it adds without ever correcting
(Yang, 2024, the blog *DeltaNet Explained*). The cell below checks the gradient with finite differences, and checks that
with $\beta = 1$ and $\lVert k \rVert = 1$, the key just written reads out exactly the value just written.

In [ ]:
rng = np.random.default_rng(1)
S0 = rng.standard_normal((16, 16))
k, v = unit(rng.standard_normal(16)), rng.standard_normal(16)
loss = lambda S: 0.5 * np.sum((k @ S - v) ** 2)
grad = np.outer(k, k @ S0 - v)  # ∇_S L = kᵀ(kS − v)
h, i, j = 1e-6, 3, 5
E = np.zeros_like(S0)
E[i, j] = h
fd = (loss(S0 + E) - loss(S0 - E)) / (2 * h)
print(f"∂L/∂S[{i},{j}]: formula {fmt(grad[i, j], 6)} · finite differences {fmt(fd, 6)}")
for beta in (1.0, 0.5):
    S1 = S0 - beta * grad
    assert np.allclose(S1, S0 + beta * np.outer(k, v - k @ S0))  # one gradient step = the delta rule
    print(f"β = {fmt(beta, 1)}: error on the pair just written {fmt(loss(S0), 3)} → {fmt(loss(S1), 3)}")
assert abs(fd - grad[i, j]) < 1e-6 and loss(S0 - grad) < 1e-20

With $\beta = 1$, the error on the pair just written drops to exactly 0: the delta rule *erases* the old value of
$k_t$, then *writes* the new one. With $\beta = 0.5$, the error shrinks to exactly $(1 - \beta)^2 = \tfrac14$ of what it
was. But each step looks at only one pair. If we make many passes over the same $N$ pairs with a small $\beta$, that is
SGD on the regression problem $\min_S \sum_i \lVert k_i S - v_i \rVert^2$, and it moves gradually toward the
least-squares solution of section 5. The cell below tries this with 32 random pairs, on all 400 draws.

In [ ]:
N, PASSES = 32, 30
K32, V32 = keys[:, :N], vals[:, :N]
S_ls = write("ls", K32, V32)
mse = lambda S: 0.5 * ((K32 @ S - V32) ** 2).sum(-1).mean()
acc32 = lambda S: read_back(S, code, K32, ids[:, :N]).mean()
curves = {}
for beta in (1.0, 0.3, 0.1):
    S = np.zeros((len(K32), 16, 16))
    curve = [mse(S)]
    for _ in range(PASSES):
        for i in range(N):
            k = K32[:, i]
            S = S + beta * k[:, :, None] * (V32[:, i] - np.einsum("bi,bij->bj", k, S))[:, None, :]
        curve.append(mse(S))
    curves[beta] = curve
    print(
        f"β = {fmt(beta, 1)}: error after 1 pass {fmt(curve[1], 3)}, after {PASSES} passes {fmt(curve[-1], 3)}; "
        f"{pct(acc32(S), 1)} read correctly"
    )
S_add = write("add", K32, V32)
print(f"least squares: error {fmt(mse(S_ls), 3)}, {pct(acc32(S_ls), 1)} correct")
print(f"S = 0: error {fmt(mse(0 * S_ls), 3)} · add: error {fmt(mse(S_add), 3)}, {pct(acc32(S_add), 1)} correct")
assert curves[0.1][-1] < curves[0.3][-1] < curves[1.0][-1] and curves[0.1][-1] - mse(S_ls) < 0.01

The figure below shows the mean squared error over the 32 pairs after each pass, for three values of $\beta$
(different colors and markers); the black dotted line is the least-squares solution, the floor that no linear way of
writing can go below.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
for (beta, curve), (color, marker) in zip(curves.items(), ((BLUE, "o"), (ORANGE, "^"), (GREEN, "D"))):
    ax.plot(
        np.arange(PASSES + 1),
        curve,
        color=color,
        marker=marker,
        ms=4,
        markevery=3,
        lw=1.8,
        label=f"delta, β = {fmt(beta, 1)}",
    )
ax.axhline(mse(S_ls), color="black", ls=":", lw=1.5, label="least squares")
ax.set(xlabel="number of passes over the 32 pairs", ylabel="mean ½‖kS − v‖²", xlim=(0, PASSES), ylim=(0, 0.6))
ax.set_title("The delta rule repeated over many passes = SGD on the key → value regression")
ax.legend(frameon=False, ncols=2)
ax.grid(alpha=0.3)
fmt_axes(fig)
plt.show()

With $\beta = 1$, each step wipes out the error of the pair being written, so it chases the newest pair and never
converges (the error oscillates around the level of $S = 0$). A smaller $\beta$ is slower but moves toward the floor: a
learning rate that is too large keeps SGD from converging, just as in Lesson 05. A DeltaNet layer makes only **one
pass** over the sequence, so it never reaches the least-squares solution; it learns $\beta_t$ to decide how much each
token is worth correcting.

### When a key is written again

Where the delta rule really makes a difference is when **the same key** is written a second time with a new value.
Write 16 pairs, then write 8 of those keys again with new values; the correct answer is the *latest* value. Adding
keeps both values superimposed, while the delta rule subtracts the old value before writing.

In [ ]:
rng = np.random.default_rng(0)
D, N, m = len(keys), 16, 8
pick = np.argsort(rng.random((D, N)), 1)[:, :m]  # the 8 keys that will be written again
new_ids = rng.integers(0, CODEWORDS, (D, m))
K_new = np.take_along_axis(keys[:, :N], pick[..., None], 1)
V_new = np.take_along_axis(code, new_ids[..., None], 1)
target = ids[:, :N].copy()
np.put_along_axis(target, pick, new_ids, 1)
is_new = np.zeros((D, N), bool)
np.put_along_axis(is_new, pick, True, 1)
for rule in ("add", "delta"):
    S = write(rule, np.concatenate([keys[:, :N], K_new], 1), np.concatenate([vals[:, :N], V_new], 1))
    got = (keys[:, :N] @ S @ code.transpose(0, 2, 1)).argmax(-1)
    ok = got == target
    old = (got == ids[:, :N])[is_new].mean()
    print(
        f"{RULES[rule]:<14}: {pct(ok.mean(), 1)} correct · rewritten keys {pct(ok[is_new].mean(), 1)} "
        f"(still read out the old value: {pct(old, 1)}) · unchanged keys {pct(ok[~is_new].mean(), 1)}"
    )

Adding doesn't know which value is newer: for a rewritten key, it reads out the new value (36.0%) about as often as
the old one (36.7%). The delta rule erases the old value before writing, so only 2.0% of the rewritten keys still read
out the old value, and 86.4% read out the new one. The price is that the *unchanged* keys drop from 67.1% to 41.8%:
every correction along the direction of $k_t$ also distorts the old keys that overlap that direction. These two
effects are exactly what training has to balance, by learning $\beta_t$ and by learning keys that overlap less.

## 7 · After training: memory still has a ceiling

The untrained memory uses random keys. A trained layer learns its own keys, values, queries and gates. Can it break
through that ceiling? The file `linear-recall.json` holds the results of exactly that problem:
- each sequence is $N$ (key, value) pairs with distinct keys, chosen from 64 keys and 64 values (each token is one-hot
  encoded);
- a single token mixer reads the whole sequence, then each key is queried once;
- each configuration trains for 6,000 steps (Adam, a batch of 64 new sequences at each step) with 3 seeds. The learning
  rate $\eta$ is chosen by seed 0 on 1,000 separate validation sequences, and accuracy is measured on 1,000 other test
  sequences.

Seven token mixers are compared: softmax attention (keeps every pair in its cache), the original linear attention
(elu + 1, with a denominator), an ungated plain sum, a forget gate, Mamba-2 (SSD), the delta rule and the gated delta
rule (Gated DeltaNet). The last six read through a $16 \times 16$ state.

### 🤔 Predict first
After training, with **64 pairs**, about what percentage does the best layer with a $16 \times 16$ state get right? And
softmax attention?

In [ ]:
ARCHS = {
    "softmax": "softmax (KV cache)",
    "gated": "forget gate",
    "gdelta": "gated delta",
    "delta": "delta rule",
    "ssd": "Mamba-2 (SSD)",
    "add": "plain sum",
    "linear": "original linear (elu + 1)",
}
N_TRAINED = (4, 8, 16, 32, 64)


def accs(task, arch, N, dk=16, fixed=False):
    """Test accuracy of the 3 seeds, read from the demo's results file."""
    run = REC["runs"][f"{task}-{arch}{'-fixed' if fixed else ''}-N{N}-d{dk}"]
    return np.array([s["acc"] for s in run["seeds"]])


def span(v, digits=1):
    """The min–max range across the seeds, as in 79.0–79.9%."""
    lo, hi = fmt(100 * v.min(), digits), fmt(100 * v.max(), digits)
    return f"{lo}%" if lo == hi else f"{lo}–{hi}%"


print("number of pairs N:        " + "".join(f"{N:>14}" for N in N_TRAINED))
for arch, label in ARCHS.items():
    print(f"{label:<26}" + "".join(f"{span(accs('recall', arch, N)):>14}" for N in N_TRAINED))
best3 = ("gated", "delta", "gdelta")
hold = [N for N in N_TRAINED if min(accs("recall", a, N).min() for a in best3) >= 0.99]
pool = lambda N: np.concatenate([accs("recall", a, N) for a in best3])
print(
    f"\nthe three gated / delta layers: ≥ 99% up to N = {max(hold)}; 32 pairs {span(pool(32))} (≈ {span(pool(32), 0)}); "
    f"64 pairs {span(pool(64))} (≈ {span(pool(64), 0)})"
)
print(
    f"original linear: 32 pairs {span(accs('recall', 'linear', 32), 0)}, 64 pairs {span(accs('recall', 'linear', 64), 0)}"
)
print(f"untrained memory, least squares: 32 pairs {pct(table['ls'][31], 1)}, 64 pairs {pct(table['ls'][63], 1)}")
soft = np.concatenate([accs("recall", "softmax", N) for N in N_TRAINED])
print(
    f"softmax: {span(soft, 0)} at every N; at 64 pairs the cache holds 64 × 32 = {fmt(64 * 32, 0)} numbers, "
    f"{64 * 32 // 256} times the state"
)

The figure below shows the same table: test accuracy against the number of pairs $N$ (horizontal axis on a base-2 log
scale), the mean of 3 seeds, with vertical bars for the min–max range. Each layer has its own color and marker; the
two thin dotted lines are section 5's **untrained** memory (add and least squares), for comparison.

In [ ]:
TRAINED_STYLE = {
    "softmax": (BLUE, "o", "-"),
    "gated": (ORANGE, "^", "-"),
    "gdelta": (GREEN, "D", "-"),
    "linear": (GREY, "s", "--"),
}
fig, ax = plt.subplots(figsize=(8, 4))
x = np.array(N_TRAINED)
for arch, (color, marker, ls) in TRAINED_STYLE.items():
    A = 100 * np.array([accs("recall", arch, N) for N in N_TRAINED])
    m = A.mean(1)
    ax.errorbar(
        x, m, yerr=[m - A.min(1), A.max(1) - m], color=color, marker=marker, ls=ls, lw=1.8, capsize=3, label=ARCHS[arch]
    )
for rule, color in (("ls", "black"), ("add", GREY)):
    ax.plot(x, 100 * table[rule][x - 1], color=color, ls=":", lw=1.2, label=f"untrained: {RULES[rule]}")
ax.set_xscale("log", base=2)
ax.set(xlabel="number of pairs N", ylabel="correct on the test set (%)", ylim=(0, 103))
ax.set_title("Key–value recall after 6,000 steps, 16 × 16 state (3 seeds)")
ax.legend(frameon=False, fontsize=8.5, loc="lower left")
ax.grid(alpha=0.3)
fmt_axes(fig)
ax.set_xticks(x, [str(n) for n in N_TRAINED])  # after fmt_axes: ticks exactly at the trained N values
ax.xaxis.set_minor_locator(NullLocator())
plt.show()

After training, the layers with a gate or the delta rule are ≥ 99% correct up to 8 pairs, then drop to 79–80% at 32
pairs and 48–49% at 64 pairs. All five fixed-state layers (plain sum, gated, Mamba-2, delta, gated delta) nearly
coincide: on this task, the way memory is managed can't rescue a state that is too small. The original linear
attention is lower still (66% and 37%). Softmax is 100% correct at every $N$, because it compresses nothing: with 64
pairs, its cache is 8 times the size of the state.

Where are the errors? The results file stores the accuracy of each question in writing order (seed 0, 32 pairs). The
cell below splits them into 4 groups from old to new, as in section 5, and prints the mean gate $\alpha$ that the gated
model learned by itself on this task.

In [ ]:
for arch in ("gated", "add", "gdelta"):
    byq = np.array(REC["runs"][f"recall-{arch}-N32-d16"]["seeds"][0]["byq"])
    print(f"{ARCHS[arch]:<14} old → new: " + " / ".join(pct(g.mean(), 1) for g in np.array_split(byq, 4)))
g32 = REC["inspect"]["recall-gated-N32"]
print(
    f"gated: mean α {fmt(g32['alphaMean'], 5)}; the first pair keeps {pct(g32['alphaPow'], 1)} of its magnitude at "
    "the end of the sequence"
)

In the gated layer, the errors are spread evenly over every group (78–80% per group): the model learns
$\alpha \approx 0.99996$ by itself, i.e. it forgets almost nothing, because forgetting here only loses information. The
errors are **overlap**, not forgetting. Gated delta, on the other hand, favors the newer pairs, like the untrained
memory in section 5: at each write, it corrects the part of the new key that coincides with the old keys.

Why can't training break through the ceiling? Part of the reason is geometry: the model can choose its keys, but it
has to place **64 keys** (one for each key code) in **16 dimensions**, so they can't be orthogonal to one another. The
Welch bound (1974) says that for $n$ unit vectors in $\mathbb{R}^d$, the mean of $\cos^2$ between two different vectors
is no smaller than $(n/d - 1)/(n - 1)$. The cell below compares that bound with random keys, and reads from the results
file how much the queries the model learned overlap.

In [ ]:
n_keys, d_k = 64, 16
welch = (n_keys / d_k - 1) / (n_keys - 1)
G = keys[:, :n_keys] @ keys[:, :n_keys].transpose(0, 2, 1)
off = G[:, ~np.eye(n_keys, dtype=bool)]
print(
    f"64 unit vectors in 16 dimensions: mean cos² ≥ {fmt(welch, 4)} (Welch bound); random keys: {fmt((off**2).mean(), 4)}"
)
print(
    f"i.e. a root-mean-square cos of about {fmt(np.sqrt(welch), 3)} at best, "
    f"against {fmt(np.sqrt((off**2).mean()), 3)} for random keys"
)
qc, phi = REC["inspect"], REC["phi"]
print(
    f"learned queries (32 pairs, seed 0): mean |cos| {fmt(qc['qcos-gated']['abs'], 3)} (gated), "
    f"{fmt(qc['qcos-add']['abs'], 3)} (plain sum); random vectors: {fmt(phi['absCosL2'], 3)}"
)
lq = qc["qcos-linear"]
print(
    f"learned queries of the original linear, after φ: mean cos {fmt(lq['mean'], 3)}, |cos| {fmt(lq['abs'], 3)} "
    "(every cos is positive)"
)
print(
    f"φ = elu + 1 is always positive: two random keys have a mean cos of {fmt(phi['cosElu'], 2)} "
    f"(smallest {fmt(phi['cosEluMin'], 2)})"
)
assert abs(welch - 3 / 63) < 1e-12

Random keys have a mean $\cos^2$ of $1/d_k = 0.0625$; the best possible keys only get down to $0.0476$. The model learns
queries that overlap a little less (mean |cos| 0.169 against 0.203), but it can't make them orthogonal. This number
can't be compared directly with the 0.218 above: that one is the root mean square of the cos, which is always at least
the mean |cos|.

The original linear attention is different in *sign*. $\varphi = \operatorname{elu} + 1$ is always positive, so two
random keys have a mean cos of 0.70. Training pulls that number down (the learned queries are left with a cos of only
0.128), but every product $\varphi(q)\cdot\varphi(k_j)$ is still positive. The output $o = \sum_j w_j v_j / \sum_j w_j$
is then a positively weighted average of *all* the values written: the noise from the other pairs adds up with the
same sign, instead of canceling out as it does with signed keys. That is why modern layers drop $\varphi$ and the
denominator, so that $q, k$ carry a sign.

Memory grows with $d_k$. The results file has the same 32-pair task with $8 \times 8$ and $32 \times 32$ states:

In [ ]:
for dk in (8, 16, 32):
    row = " · ".join(
        f"{ARCHS[a]} {span(accs('recall', a, 32, dk), 2)}" for a in ("gated", "gdelta", "linear", "softmax")
    )
    print(f"{dk:>2} × {dk:<2} ({fmt(dk * dk, 0):>5} numbers), 32 pairs: {row}")
print("each seed, forget gate at 8 × 8:", " · ".join(pct(x, 1) for x in accs("recall", "gated", 32, 8)))
small = np.concatenate([accs("recall", a, 32, 8) for a in ("gated", "gdelta", "linear")])
print(f"8 × 8, three layers, stuck seed left out: {span(small[small > 0.1], 0)}; chance: 1/64 = {pct(1 / 64, 1)}")

With 4 times as many entries (32 × 32), the 32 pairs are almost all correct again (99.98–99.99%); with a quarter as many
(8 × 8), only 18–23%, and one seed of the gated layer even gets stuck at 2.4%, near the chance level of 1/64. Softmax
is 100% correct at all three sizes, because it doesn't rely on $d_k$ to *store*, only to *distinguish*.

### Is the ceiling in the matrix, or in the way of writing?

An easily overlooked detail: least squares gives the best matrix in **squared error**, not in **recall accuracy**. If
we are free to choose $S$ so that each $k_i S$ is closer to the right word *than to every other word* (optimizing the
cross-entropy of Lesson 04 over the 64 words, with Adam, starting from the least-squares solution), how many pairs can a
$16 \times 16$ matrix hold? The cell below tries it on the first 100 draws.

In [ ]:
def fit_readout(K, V, c, y, steps=300, lr=0.01, scale=8.0):
    """Find S so that argmax_w (k_i S)·c_w = y_i, with Adam on the cross-entropy; starts from the least-squares
    solution."""
    S = write("ls", K, V)
    m1, m2 = np.zeros_like(S), np.zeros_like(S)
    for t in range(1, steps + 1):
        z = scale * (K @ S @ c.transpose(0, 2, 1))
        P = np.exp(z - z.max(-1, keepdims=True))
        P /= P.sum(-1, keepdims=True)
        np.put_along_axis(P, y[..., None], np.take_along_axis(P, y[..., None], -1) - 1, -1)  # softmax − one-hot
        g = scale * K.transpose(0, 2, 1) @ P @ c / K.shape[1]
        m1, m2 = 0.9 * m1 + 0.1 * g, 0.999 * m2 + 0.001 * g**2
        S -= lr * (m1 / (1 - 0.9**t)) / (np.sqrt(m2 / (1 - 0.999**t)) + 1e-8)
    return read_back(S, c, K, y).mean()


D = 100
for N in (32, 64):
    acc_fit = fit_readout(keys[:D, :N], vals[:D, :N], code[:D], ids[:D, :N])
    acc_ls = read_back(write("ls", keys[:D, :N], vals[:D, :N]), code[:D], keys[:D, :N], ids[:D, :N]).mean()
    print(f"{N} pairs, 100 draws: least squares {pct(acc_ls, 1)} → optimized for recall accuracy {pct(acc_fit, 1)}")

With the same $16 \times 16$ matrix, 32 pairs are **100%** correct when $S$ is optimized specifically for reading back,
against 78.3% for least squares and 79–80% for the trained layers. At 64 pairs, 66.4% against 48–49%. So the ceiling
above isn't really the capacity of the matrix. It is a limit of the **way of writing**: a linear layer makes only one
pass over the sequence, adding up rank-1 updates, and can't solve an optimization problem like the cell above while it
reads. That is the direction of TTT-style layers (Sun et al., 2024): treat the state as a small model that is *learned*
on the fly, while reading. The delta rule is exactly the simplest case, one gradient step per token.

## 8 · Training a forget gate ourselves

The layers above didn't need to forget yet: each sequence had only one group of pairs. The **reset** task (erasing
the old segment) does need it. Each sequence consists of 16 pairs, a separator, then 16 *other* pairs, and only the
keys of the second group are queried. The first group only fills up the state. A model that knows to erase the state
at the separator has only 16 pairs left to remember; one that doesn't has to put up with the overlap of all 32.

We write a layer with a forget gate ourselves, in exactly the configuration of the results file (the course's
`reel/linear/mixers.py`). Each token $x$ is a vector made of the one-hot key code (64), the one-hot value code (64) and
a flag (1 at the separator). From it:

$$k = \frac{xW_k}{\lVert xW_k \rVert},\quad v = xW_v,\quad \alpha = \sigma(xw_\alpha + b_\alpha),\quad \beta = \sigma(xw_\beta + b_\beta),
\qquad S_T = \sum_{s} \Big(\prod_{r > s} \alpha_r\Big)\beta_s\, k_s^\top v_s$$

Each question is a learned query $q = W_q[\text{key code}]$ (normalized to length 1), which reads $o = qS_T$; then a
linear layer $oW_o + b_o$ gives 64 scores, followed by a softmax cross-entropy (Lesson 04). With a **constant gate**,
$\alpha$ and $\beta$ don't depend on the token: $w_\alpha = w_\beta = 0$, and only $b_\alpha, b_\beta$ are learned. The
cell below generates data exactly as `recall.py` does.

In [ ]:
NK = NV = 64  # number of key codes and value codes
F = NK + NV + 1  # one-hot key, one-hot value, flag


def features(keys_id, vals_id, flag):
    """Code −1 (the separator) turns on no one-hot entry."""
    B_, T_ = keys_id.shape
    X = np.zeros((B_, T_, F))
    b_, t_ = np.nonzero(keys_id >= 0)
    X[b_, t_, keys_id[b_, t_]] = 1
    b_, t_ = np.nonzero(vals_id >= 0)
    X[b_, t_, NK + vals_id[b_, t_]] = 1
    X[..., -1] = flag
    return X


def make_task(rng, n, N, task):
    """n sequences → X (n, T, F), the key codes queried (n, N) and the answers (n, N); the same as the course's
    recall.py."""
    ks = np.argsort(rng.random((n, NK)), 1)[:, :N]  # N distinct key codes
    vs = rng.integers(0, NV, (n, N))
    if task == "recall":  # N pairs, query them all
        return features(ks, vs, np.zeros((n, N))), ks.copy(), vs
    # reset: N pairs, a separator, N pairs with new keys; query the second group (recall.py draws ks, vs above first,
    # as here)
    ks = np.argsort(rng.random((n, NK)), 1)[:, : 2 * N]
    v1, v2 = rng.integers(0, NV, (n, N)), rng.integers(0, NV, (n, N))
    sep = np.full((n, 1), -1)
    kk, vv = np.concatenate([ks[:, :N], sep, ks[:, N:]], 1), np.concatenate([v1, sep, v2], 1)
    flag = np.zeros(kk.shape)
    flag[:, N] = 1
    return features(kk, vv, flag), ks[:, N:].copy(), v2


X, Qk, Y = make_task(np.random.default_rng(2016), 1000, 16, "reset")  # the results file's test set
strip = REC["inspect"]["reset-gated"]["strip"]
key0 = [int(np.argmax(t[:NK])) if t[:NK].any() else -1 for t in X[0]]
print(f"X has shape {X.shape}; first test sequence: key codes {key0[:6]} … separator at position {key0.index(-1)}")
assert key0 == strip["key"], "the data generator must match recall.py"

The model and backpropagation, written by hand (Lesson 06). Since only the final state is read, $S_T$ is a weighted
sum with weights $e_s = w_s \beta_s$, where $w_s = \prod_{r>s}\alpha_r = e^{c_T - c_s}$, so no loop over time is needed:
the gradient with respect to $\log\alpha$ goes through the cumulative sum $c$. Three kinds of gate: `"token"` (learns
$w_\alpha, w_\beta$), `"fixed"` (only $b_\alpha, b_\beta$), `"none"` ($\alpha = \beta = 1$, a plain sum).

In [ ]:
def sigmoid(z):
    return 0.5 * (1 + np.tanh(0.5 * z))  # the same value, with no overflow when |z| is large


def init(dk, seed):
    """As in the course's mixers.init: b_α = 3 so that α ≈ 0.95 at first; b_β = 0 so that β = 0.5."""
    r = np.random.default_rng(seed)
    Wk, Wv = r.standard_normal((F, dk)) / np.sqrt(3), r.standard_normal((F, dk)) / np.sqrt(3)
    Wq, Wo = r.standard_normal((NK, dk)), r.standard_normal((dk, NV)) / np.sqrt(dk)
    zero = np.zeros(F)
    return {"Wk": Wk, "Wv": Wv, "wa": zero, "ba": np.array([3.0]), "wb": zero.copy(), "bb": np.array([0.0]),
            "Wq": Wq, "Wo": Wo, "bo": np.zeros(NV)}  # fmt: skip


def l2n(x):
    n = np.sqrt((x * x).sum(-1, keepdims=True) + 1e-12)
    return x / n, n


def forward(p, X, Qk, gate):
    B_, T_, _ = X.shape
    k, kn = l2n(X @ p["Wk"])
    q, qn = l2n(p["Wq"][Qk])
    v = X @ p["Wv"]
    if gate == "none":
        za, la, b = None, np.zeros((B_, T_)), np.ones((B_, T_))
    else:
        tok = gate == "token"
        za = X @ p["wa"] + p["ba"][0] if tok else np.full((B_, T_), p["ba"][0])
        zb = X @ p["wb"] + p["bb"][0] if tok else np.full((B_, T_), p["bb"][0])
        la, b = -np.logaddexp(0, -za), sigmoid(zb)  # log α = log σ(z_α), written stably
    c = np.cumsum(la, 1)
    w = np.exp(c[:, -1:] - c)  # ∏_{r>s} α_r: the share of token s that is left at the end of the sequence
    S = (k * (w * b)[..., None]).transpose(0, 2, 1) @ v  # S_T = Σ_s w_s β_s k_sᵀ v_s
    o = q @ S
    return {"k": k, "kn": kn, "q": q, "qn": qn, "v": v, "za": za, "la": la, "b": b, "w": w, "S": S, "o": o,
            "z": o @ p["Wo"] + p["bo"]}  # fmt: skip

In [ ]:
def l2n_back(g, y, n):
    """The gradient through y = x/‖x‖: remove the component parallel to y, then divide by ‖x‖."""
    return (g - y * (g * y).sum(-1, keepdims=True)) / n


def loss_grad(p, X, Qk, Y, gate):
    f = forward(p, X, Qk, gate)
    z = f["z"] - f["z"].max(-1, keepdims=True)
    P = np.exp(z)
    P /= P.sum(-1, keepdims=True)
    loss = -np.log(np.take_along_axis(P, Y[..., None], -1)).mean()
    dz = P  # ∂L/∂z = (softmax − one-hot) / number of questions
    np.put_along_axis(dz, Y[..., None], np.take_along_axis(dz, Y[..., None], -1) - 1, -1)
    dz /= Y.size
    k, q, v, w, b, S, o = (f[n] for n in ("k", "q", "v", "w", "b", "S", "o"))
    B_, T_, dk = k.shape
    g = {"Wo": o.reshape(-1, dk).T @ dz.reshape(-1, NV), "bo": dz.sum((0, 1))}
    do = dz @ p["Wo"].T
    G = q.transpose(0, 2, 1) @ do  # ∂L/∂S_T
    dq = l2n_back(do @ S.transpose(0, 2, 1), q, f["qn"])
    e = w * b
    kG = k @ G
    dk_ = e[..., None] * (v @ G.transpose(0, 2, 1))
    dv = e[..., None] * kG
    de = (kG * v).sum(-1)  # ∂L/∂e_s
    db, dlw = de * w, de * e  # e = w·β; ∂L/∂log w_s = ∂L/∂e_s · e_s
    dc = -dlw  # log w_s = c_T − c_s
    dc[:, -1] += dlw.sum(1)
    dla = np.cumsum(dc[:, ::-1], 1)[:, ::-1]  # c_t = Σ_{u≤t} log α_u
    if gate == "none":
        dza = dzb = np.zeros((B_, T_))
    else:
        dza, dzb = dla * (1 - sigmoid(f["za"])), db * b * (1 - b)  # d log σ(z)/dz = 1 − σ(z); dσ/dz = σ(1 − σ)
    Xf = X.reshape(B_ * T_, F)
    g["Wk"] = Xf.T @ l2n_back(dk_, k, f["kn"]).reshape(-1, dk)
    g["Wv"] = Xf.T @ dv.reshape(-1, dk)
    tok = gate == "token"
    g["wa"], g["wb"] = (Xf.T @ dza.ravel(), Xf.T @ dzb.ravel()) if tok else (np.zeros(F), np.zeros(F))
    g["ba"], g["bb"] = np.array([dza.sum()]), np.array([dzb.sum()])
    g["Wq"] = np.zeros_like(p["Wq"])
    np.add.at(g["Wq"], Qk.ravel(), dq.reshape(-1, dk))
    return loss, g


# finite-difference gradient check, on a small batch with a nonzero per-token gate
rng = np.random.default_rng(3)
p = init(16, 1)
p["wa"], p["wb"] = rng.standard_normal(F) * 0.5, rng.standard_normal(F) * 0.5
Xs, Qs, Ys = make_task(rng, 4, 4, "reset")
_, g = loss_grad(p, Xs, Qs, Ys, "token")
worst = 0.0
for name in p:
    for idx in [np.unravel_index(j * 37 % p[name].size, p[name].shape) for j in range(4)]:
        old = p[name][idx]
        p[name][idx] = old + 1e-6
        lp = loss_grad(p, Xs, Qs, Ys, "token")[0]
        p[name][idx] = old - 1e-6
        lm = loss_grad(p, Xs, Qs, Ys, "token")[0]
        p[name][idx] = old
        num = (lp - lm) / 2e-6
        worst = max(worst, abs(num - g[name][idx]) / max(1e-8, abs(num) + abs(g[name][idx])))
print(f"largest relative error between the handwritten gradient and finite differences: {sci(worst)}")
assert worst < 1e-5

The handwritten gradient matches the finite differences. The training loop is the same as `recall.py`'s: Adam
($0.9$ / $0.999$), a batch of 64 new sequences at each step, linear warmup followed by cosine decay, gradient-norm
clipping at 1. The results file trained for 6,000 steps; here we train for only 800 (100 warmup steps instead of 200)
with the $\eta$ the results file chose on the validation set, enough to see what the gate learns (a few seconds per run).

In [ ]:
def train(task, N, gate, lr, steps=800, warm=100, seed=0, B=64):
    p = init(16, seed + 1)
    m1 = {n: np.zeros_like(x) for n, x in p.items()}
    m2 = {n: np.zeros_like(x) for n, x in p.items()}
    rng = np.random.default_rng(seed + 100)  # training data: always new sequences
    for s in range(1, steps + 1):
        X, Qk, Y = make_task(rng, B, N, task)
        _, g = loss_grad(p, X, Qk, Y, gate)
        gn = np.sqrt(sum((x**2).sum() for x in g.values()))
        if gn > 1:  # gradient-norm clipping
            g = {n: x / gn for n, x in g.items()}
        lr_s = lr * min(1, s / warm) * 0.5 * (1 + np.cos(np.pi * s / steps))
        for n in p:
            m1[n] = 0.9 * m1[n] + 0.1 * g[n]
            m2[n] = 0.999 * m2[n] + 0.001 * g[n] ** 2
            p[n] -= lr_s * (m1[n] / (1 - 0.9**s)) / (np.sqrt(m2[n] / (1 - 0.999**s)) + 1e-8)
    return p


def test_acc(p, gate, task="reset", N=16):
    X, Qk, Y = make_task(np.random.default_rng(2000 + N), 1000, N, task)  # the results file's 1,000 test sequences
    f = forward(p, X, Qk, gate)
    return (f["z"].argmax(-1) == Y).mean(), f, X


LR = {g: REC["runs"][f"reset-gated{'' if g == 'token' else '-fixed'}-N16-d16"]["lr"] for g in ("token", "fixed")}
models, live = {}, {}
for gate in ("token", "fixed"):
    models[gate] = train("reset", 16, gate, LR[gate])
    acc, f, Xt = test_acc(models[gate], gate)
    alpha = np.exp(f["la"])
    live[gate] = (acc, alpha, f["b"])
    print(
        f"{'per-token' if gate == 'token' else 'constant':<9} gate (η = {fmt(LR[gate], 2)}): {pct(acc, 1)} correct · "
        f"α at the separator {fmt(alpha[:, 16].mean(), 3)}, at other tokens {fmt(np.delete(alpha, 16, 1).mean(), 4)}"
    )
ins = REC["inspect"]["reset-gated"]
print(
    f"results file (6,000 steps, seed 0): α at the separator {fmt(ins['alphaSep'], 4)}, at other tokens "
    f"{fmt(ins['alphaOther'], 4)}; α of the constant gate {fmt(ins['fixedAlpha'], 3)}"
)
print(
    f"3 seeds: per-token {span(accs('reset', 'gated', 16))} · constant {span(accs('reset', 'gated', 16, fixed=True))}"
)
print(f"no gate (plain sum): {span(accs('reset', 'add', 16))} · softmax: {span(accs('reset', 'softmax', 16))}")
acc_tok, a_tok, _ = live["token"]
a_fix = live["fixed"][1][0, 0]
print(
    f"constant gate α = {fmt(a_fix, 3)}: at the end of the sequence, the first group keeps {fmt(a_fix**33, 2)}–"
    f"{fmt(a_fix**17, 2)} of its magnitude, the second group {fmt(a_fix**15, 2)}–1"
)
assert acc_tok > live["fixed"][0] + 0.05 and a_tok[:, 16].mean() < 0.05 and np.delete(a_tok, 16, 1).mean() > 0.99

After 800 steps, the model with a per-token gate has learned by itself to **erase**: $\alpha$ at the separator has
dropped to 0.016, while at every other token $\alpha \approx 0.998$, i.e. it forgets almost nothing. Nobody taught it
what the separator means; the loss only measures the answers about the second group. It is 96.6% correct, close to the
level of the 16-pair task without a first group (96.9–97.1% in section 7), so the first group leaves almost no noise
behind. The constant gate can only pick one $\alpha \approx 0.966$ for every token, a compromise: the first group only
shrinks to 0.32–0.55 of its magnitude, while the oldest pair of the second group has already faded to 0.59. The results
file, trained for 6,000 steps, gives the same picture, with an even smaller $\alpha$ at the separator (0.0036).

The figure below shows $\alpha_t$ and $\beta_t$ by token position on the first test sequence: blue circles are the
model just trained above, orange squares the results file's seed-0 model. The gray bar is the separator.

In [ ]:
_, f_tok, _ = test_acc(models["token"], "token")
a_live, b_live = np.exp(f_tok["la"][0]), f_tok["b"][0]
pos = np.arange(len(a_live))
fig, axes = plt.subplots(2, 1, figsize=(9, 4.4), sharex=True)
for ax, live_v, saved, name in (
    (axes[0], a_live, strip["alpha"], "αₜ (forget gate)"),
    (axes[1], b_live, strip["beta"], "βₜ (write strength)"),
):
    ax.axvline(16, color=GREY, lw=6, alpha=0.3)
    ax.plot(pos, live_v, "o-", color=BLUE, ms=5, lw=1.2, label="just trained (800 steps)")
    ax.plot(pos, saved, "s", color=ORANGE, ms=4, mfc="none", mew=1.4, label="results file (6,000 steps)")
    ax.set(ylabel=name, ylim=(-0.05, 1.08))
    ax.grid(alpha=0.3)
axes[0].legend(frameon=False, loc="lower left", ncols=2)
axes[0].set_title("Gates the model learned by itself on the reset task: 16 pairs | separator | 16 pairs")
axes[1].set_xlabel("token position (16 = separator)")
plt.tight_layout()
fmt_axes(fig)
plt.show()
print(f"β at the separator: just trained {fmt(b_live[16], 3)}, results file {fmt(ins['betaSep'], 3)}")
print(
    f"β at the real pairs: just trained ≈ {fmt(np.delete(b_live, 16).mean(), 2)}, results file ≈ {fmt(ins['betaOther'], 2)}"
)

$\beta_t$ also switches itself off at the separator (there is nothing to write). The two models learn different
$\beta$ at the real pairs (0.93 against 0.74), because the overall magnitude of $S$ can be shifted into $W_v$ or $W_o$:
only the shape of the curve is meaningful.

### Overwriting a key, and filtering out noise

Two harder tasks in the results file (16 pairs, the same $16 \times 16$ state, 3 seeds):
- **overwrite:** 16 pairs, then 8 of those keys are written again with new values; the correct answer is the latest
  value (like the untrained experiment in section 6);
- **noise:** 16 real pairs interleaved with 16 flagged noise tokens with random keys and values; the noise has to be
  ignored.

“Positional” softmax (`softpos`) is softmax plus a learned positional slope (ALiBi-style), so it knows which token
came later.

In [ ]:
OVERWRITE = {"add": "plain sum", "gated": "gated", "delta": "delta rule", "gdelta": "gated delta",
             "linear": "original linear", "softmax": "softmax", "softpos": "softmax with positions"}  # fmt: skip
print(f"{'rewrite 8 of 16 keys':<24}{'all':>14}{'rewritten keys':>16}{'unchanged keys':>17}")
for arch, label in OVERWRITE.items():
    seeds = REC["runs"][f"overwrite-{arch}-N16-d16"]["seeds"]
    new, kept = (np.array([s[k] for s in seeds]) for k in ("accNew", "accKept"))
    print(f"{label:<24}{span(accs('overwrite', arch, 16)):>14}{span(new):>16}{span(kept):>17}")
soft_new = np.array([s["accNew"] for s in REC["runs"]["overwrite-softmax-N16-d16"]["seeds"]])
print(
    f"rounded: delta {span(accs('overwrite', 'delta', 16), 0)}, gated {span(accs('overwrite', 'gated', 16), 0)}; "
    f"softmax: rewritten keys ≈ {span(soft_new, 0)}, all ≈ {span(accs('overwrite', 'softmax', 16), 0)}"
)
blind = 0.5 * 1.0 + 0.5 * 0.5  # half the keys unchanged (all correct), half rewritten (new value guessed half the time)
print(f"the level of a model that can't tell old values from new ones: {pct(blind, 0)}")
noise, fixed = REC["inspect"]["noise-gated"], REC["inspect"]["noise-gated-fixed"]
print(
    f"\nnoise: per-token gate {span(accs('noise', 'gated', 16))}, constant gate "
    f"{span(accs('noise', 'gated', 16, fixed=True))}, softmax {span(accs('noise', 'softmax', 16))}"
)
print(
    f"  per-token gate (seed 0): β at the noise {fmt(noise['betaNoise'], 3)}, at the real pairs {fmt(noise['betaReal'], 3)}"
)
print(
    f"  constant gate (seed 0): β = {fmt(fixed['betaReal'], 3)} for every token; |cos| between the queries and the "
    f"noise keys {fmt(fixed['cosNoise'], 3)}, the real keys {fmt(fixed['cosReal'], 3)}"
)

On the overwrite task, the delta rule (and gated delta) is 91% correct, while the forget gate gets only 78%. The gate
shrinks the *whole* state, so it can't erase the old value of one key while keeping the other keys; the delta rule
corrects exactly the direction of $k_t$. Softmax without positions keeps 100% of the unchanged keys right but only
50–51% of the rewritten keys: it sees both values and doesn't know which one is newer, so it stops at
$\tfrac12 \cdot 100\% + \tfrac12 \cdot 50\% = 75\%$. With positions added, softmax is 100% correct again. The forget
gate here is also only slightly above that 75% level.

On the noise task, the per-token gate learns $\beta \approx 0$ for the noise tokens: it doesn't write them into the
state. The constant gate can't do that, so it finds another way: it places the noise keys nearly orthogonal to every
query (|cos| 0.025 against 0.208), i.e. it sets aside part of the 16-dimensional space as a “trash bin”. That is what
Mamba added to S4: making the gate depend on the token (“selective”).

## 9 · Mamba and Mamba-2: the same computation, seen from the state-space-model side

Mamba (Gu & Dao, 2023) doesn't start from attention but from a discrete **state space model** (SSM). In the row
convention, for a head with $P$ channels ($x_t$ is $1 \times P$) and a state $h_t$ of size $N \times P$:

$$h_t = \bar A_t\, h_{t-1} + \Delta_t\, B_t^\top x_t, \qquad y_t = C_t\, h_t, \qquad \bar A_t = e^{-\Delta_t a}$$

- S4, before Mamba, used fixed $\Delta, B, C$: the model is time-invariant, so the whole layer is a causal
  **convolution** with a fixed kernel, and can be trained as a convolution.
- Mamba makes $B_t, C_t$ and the step $\Delta_t$ depend on the token (“selective”). $\bar A_t$ becomes a token-dependent
  forget gate, but the convolution can no longer be used; Mamba makes up for it with a **parallel scan** written
  specifically for GPUs. In Mamba, $A$ is a diagonal matrix: each state dimension forgets at its own rate.
- Mamba-2 (Dao & Gu, 2024) reduces $A$ to **a single number** $a$ per head. Then, renaming $(C, B, x) \to (q, k, v)$,
  $\alpha_t = e^{-\Delta_t a}$, $\beta_t = \Delta_t$, we get exactly the gated linear attention of section 4. The authors
  call this *state space duality* (SSD): the recurrent form and the matrix form $(L \odot CB^\top)x$ are the same model.

The cell below runs an SSD layer by the SSM formula itself, then runs section 4's `parallel` function with the renamed
inputs. After that, it makes $\Delta, B, C$ fixed and compares that layer with a convolution.

In [ ]:
def ssm_scan(x, Bm, Cm, dt, a):
    """Selective SSM with A = −a·I (Mamba-2): h_t = exp(−Δ_t a) h_{t−1} + Δ_t B_tᵀ x_t, y_t = C_t h_t."""
    h, y = np.zeros((Bm.shape[1], x.shape[1])), np.zeros_like(x)
    for t in range(len(x)):
        h = np.exp(-dt[t] * a) * h + dt[t] * np.outer(Bm[t], x[t])
        y[t] = Cm[t] @ h
    return y


rng = np.random.default_rng(0)
T, Nst, P, a_ssm = 64, 16, 8, 0.5
x = rng.standard_normal((T, P))
Bm, Cm = rng.standard_normal((T, Nst)), rng.standard_normal((T, Nst))
dt = np.logaddexp(0, rng.standard_normal(T) - 1)  # Δ_t = softplus(·) > 0, depends on the token
y_ssm = ssm_scan(x, Bm, Cm, dt, a_ssm)
y_attn = parallel(Cm, Bm, x, np.exp(-dt * a_ssm), dt)  # (C, B, x) → (q, k, v), α = exp(−Δa), β = Δ
print(
    f"selective SSM (Mamba-2) and gated linear attention, matrix form: differ by at most {sci(np.abs(y_ssm - y_attn).max())}"
)
assert np.abs(y_ssm - y_attn).max() < 1e-10

# fixed Δ, B, C (S4-style): y is the causal convolution of x with the kernel κ_j = Δ·(C·B)·exp(−Δa)^j
d0, b0, c0 = 0.4, Bm[0], Cm[0]
y_lti = ssm_scan(x, np.tile(b0, (T, 1)), np.tile(c0, (T, 1)), np.full(T, d0), a_ssm)
kernel = d0 * (c0 @ b0) * np.exp(-d0 * a_ssm) ** np.arange(T)
y_conv = np.stack([np.convolve(x[:, j], kernel)[:T] for j in range(P)], 1)
print(
    f"fixed parameters: the SSM and a convolution with a {T}-coefficient kernel differ by {sci(np.abs(y_lti - y_conv).max())}"
)
assert np.abs(y_lti - y_conv).max() < 1e-10

The two ways of writing it agree to rounding error. Mamba-2 *is* gated linear attention, just named in the SSM
tradition; it has an implicit attention matrix $(L \odot CB^\top)$ even though it never computes it. When the parameters
don't depend on the token, the same layer is a convolution: that is S4. “Selectivity” is what makes Mamba different
from S4, and section 8 has just shown where it pays off: when the model has to forget or ignore things *depending on
the content*.

## 10 · Hybrid models: memory when reading long texts

A fixed-size state can't recall everything exactly (section 7), while full attention costs memory (section 3). Many
recent large models keep **a few full attention layers** for recall and make the rest linear layers. The cell below
records their configurations, taken from the `config.json` published on Hugging Face (the same versions the demo
uses), then computes:
- the KV cache per token, of the attention layers only;
- the fixed-size state of the linear layers: the matrix $S$ of every head, plus the buffer of the short convolution
  (kernel 4, keeping 3 tokens) these layers apply before computing $q, k, v$.

### 🤔 Predict first
Qwen3-Next-80B-A3B has 80 billion parameters, 10 times as many as Llama 3 8B. At 262,144 tokens, is its cache larger
or smaller than Llama 3 8B's 32 GiB?

In [ ]:
HYBRID = {
    # Qwen/Qwen3-Next-80B-A3B-Instruct: 48 layers, every 4th is attention (full_attention_interval 4); attention 2 KV
    # heads × 256; Gated DeltaNet 32 value heads × (128 × 128), 16 key heads × 128, convolution kernel 4
    "Qwen3-Next-80B-A3B": {"attn": 12, "lin": 36, "kv": 2 * 2 * 256, "state": 32 * 128 * 128,
                           "conv": (2 * 16 * 128 + 32 * 128) * 3},
    # ai21labs/Jamba-v0.1: 32 layers, attention at layers 4, 12, 20, 28 (period 8, offset 4); 8 KV heads × 128 (4096 /
    # 32 heads); Mamba: 2 × 4096 = 8192 channels × d_state 16, convolution kernel 4
    "Jamba v0.1": {"attn": 4, "lin": 28, "kv": 2 * 8 * 128, "state": 8192 * 16, "conv": 8192 * 3},
    # MiniMaxAI/MiniMax-Text-01: 80 layers, 1 softmax attention every 8 layers (8 KV heads × 128); lightning attention
    # 64 heads × 128²
    "MiniMax-Text-01": {"attn": 10, "lin": 70, "kv": 2 * 8 * 128, "state": 64 * 128 * 128, "conv": 0},
    # MiniMaxAI/MiniMax-M2: 62 layers, all of them full attention (8 KV heads × 128)
    "MiniMax-M2": {"attn": 62, "lin": 0, "kv": 2 * 8 * 128, "state": 0, "conv": 0},
    # deepseek-ai/DeepSeek-V3.2: 61 MLA layers; each token in each layer keeps 512 compressed numbers + 64 RoPE key
    # numbers; each query reads only 2,048 tokens picked by the indexer (sparse attention), but every token is still kept
    "DeepSeek-V3.2": {"attn": 61, "lin": 0, "kv": 512 + 64, "state": 0, "conv": 0},
    "Llama 3 8B": {"attn": 32, "lin": 0, "kv": 2 * 8 * 128, "state": 0, "conv": 0},
    "Llama 2 7B": {"attn": 32, "lin": 0, "kv": 2 * 32 * 128, "state": 0, "conv": 0},
}  # fmt: skip


def memory(m, T, bytes_per_number=2):
    """The KV cache at T tokens and the fixed-size state, in bytes (bf16)."""
    return m["attn"] * m["kv"] * T * bytes_per_number, m["lin"] * (m["state"] + m["conv"]) * bytes_per_number


ref = {m["name"]: m for m in COST["models"]}  # the same numbers, computed from config.json by the course's cost.py
for name, m in HYBRID.items():
    kv, st = memory(m, T_LONG)
    r = ref[name]["at"][str(T_LONG)]
    same = (kv, st) == (r["kv"], r["state"])
    print(
        f"{name:<19} {m['attn']:>2} attention + {m['lin']:>2} linear: {fmt(kv / T_LONG / KiB, 1):>5} KiB per token  →  "
        f"{fmt(kv / GiB, 2):>6} GiB KV + {fmt(st / MiB, 1):>5} MiB state = {fmt((kv + st) / GiB, 2):>6} GiB  "
        f"(as in the demo's file: {same})"
    )
    assert same, "differs from the demo: check the configuration again"
assert memory(HYBRID["Jamba v0.1"], T_LONG)[0] == 4 * GiB  # Jamba's Table 1: 4GB at 256K tokens, 16 bits
assert memory(HYBRID["Qwen3-Next-80B-A3B"], 1)[0] == 24 * KiB  # 12 layers × 2 KV heads × 256 × (K and V) × 2 bytes

Qwen3-Next's cache at 262,144 tokens is only 6 GiB, plus about 38 MiB of fixed-size state: 6.04 GiB in total, less
than a fifth of Llama 3 8B's 32 GiB, even though the model is 10 times larger. Two reasons add up: only 12 of the 48
layers keep a cache, and each of those layers has only 2 KV heads. Jamba comes to exactly 4 GiB, as in Table 1 of its
paper. The fixed-size state is so small that it is barely visible.

The layer layouts, read from the demo's file (A: full attention, S: sparse attention that still keeps every token, L:
linear layer):

In [ ]:
for name in (
    "Qwen3-Next-80B-A3B",
    "Jamba v0.1",
    "MiniMax-Text-01",
    "Kimi Linear 48B",
    "Nemotron-H-8B",
    "DeepSeek-V3.2",
):
    m = ref[name]
    n_a = sum(ch in "AS" for ch in m["layout"])
    print(f"{name:<19} {m['layout']}  ({n_a} attention / {len(m['layout'])} layers; linear layer: {m['mixer'] or '—'})")
assert ref["Qwen3-Next-80B-A3B"]["layout"] == "LLLA" * 12 and ref["Jamba v0.1"]["layout"].count("A") == 4

The figure below shows the total memory (KV plus state, bf16) against the length of the text, with both axes on a log
scale. Each model has its own color and marker. On a log–log scale, memory proportional to $T$ is a straight line of
slope 1; the models differ only in the number of bytes per token (the height of the line). A model with only linear
layers would be a horizontal line.

In [ ]:
Ts = np.array([2**k for k in range(10, 21)])
LINES = {
    "Llama 2 7B": (GREY, "s", "--"),
    "MiniMax-M2": (ORANGE, "v", "-"),
    "Llama 3 8B": (BLUE, "o", "-"),
    "DeepSeek-V3.2": ("black", "x", ":"),
    "Qwen3-Next-80B-A3B": (GREEN, "D", "-"),
    "Jamba v0.1": (GREEN, "^", "--"),
}
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for name, (color, marker, ls) in LINES.items():
    total = [sum(memory(HYBRID[name], T)) / GiB for T in Ts]
    ax.plot(Ts, total, color=color, marker=marker, ls=ls, ms=5, lw=1.6, label=name)
ax.axvline(T_LONG, color=GREY, lw=1, ls=":")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(xlabel="text length T (tokens)", ylabel="KV cache + state (GiB)", xlim=(Ts[0], Ts[-1]))
ax.set_title("Memory when reading long texts, computed from config.json (bf16)")
ax.legend(frameon=False, fontsize=9, loc="upper left")
ax.grid(alpha=0.3, which="major")
fmt_axes(fig)
plt.show()
q, l3 = sum(memory(HYBRID["Qwen3-Next-80B-A3B"], T_LONG)), sum(memory(HYBRID["Llama 3 8B"], T_LONG))
print(
    f"at {fmt(T_LONG, 0)} tokens: Qwen3-Next {fmt(q / GiB, 2)} GiB, Llama 3 8B {fmt(l3 / GiB, 0)} GiB "
    f"({fmt(l3 / q, 1)} times as much)"
)
kv1, st = memory(HYBRID["Qwen3-Next-80B-A3B"], 1)
print(
    f"Qwen3-Next: the KV cache exceeds the fixed-size state ({fmt(st / MiB, 1)} MiB) from about {fmt(st / kv1, 0)} tokens"
)
ratio = {
    T: sum(memory(HYBRID["Llama 3 8B"], T)) / sum(memory(HYBRID["Qwen3-Next-80B-A3B"], T))
    for T in (1024, 4096, 16384, T_LONG)
}
print(
    "Llama 3 8B against Qwen3-Next: "
    + ", ".join(f"{fmt(r, 1)} times as much at {fmt(T, 0)} tokens" for T, r in ratio.items())
)
assert ratio[1024] < 2.5 and ratio[16384] > 4.5  # about 5 times only once the text is long enough

Qwen3-Next's line bends at short lengths: below about 1,608 tokens, the 37.7 MiB of fixed-size state is still larger
than the KV cache. From there on, the cache of the 12 attention layers makes up most of it, and the line straightens
out again. At 1,024 tokens it lies only about 2 times below Llama 3 8B; from about 16,384 tokens on, about 5 times.

Why not drop attention altogether? Section 7 showed that a fixed-size state can't recall exactly when there are many
pairs. Jelassi et al. (2024) prove something similar for copying strings: a two-layer Transformer can copy strings of
exponential length, while a model with a fixed-size state is limited by the size of its state. A few full attention
layers take care of that recall. But the race isn't settled:
- **Sparse attention:** DeepSeek-V3.2 still keeps every token in its cache, but each query reads only 2,048 tokens
  picked by an indexer. It saves computation, not memory.
- **Back to full attention:** MiniMax-M2 uses full attention in all 62 layers, after MiniMax-Text-01 used a 7 : 1
  hybrid. According to MiniMax's blog, the hybrid was on par on older benchmarks but clearly weaker at multi-step
  reasoning at scale.
- **Staying hybrid:** Kimi Linear and Qwen3.5 keep a ratio of about 3 linear layers to 1 attention layer.

In [ ]:
for name in ("Qwen3.5-397B-A17B", "Kimi Linear 48B"):
    m = ref[name]
    n_a, n_l = m["layout"].count("A"), m["layout"].count("L")
    print(f"{name:<19} {m['layout']}  ({n_l} {m['mixer']} : {n_a} attention ≈ {fmt(n_l / n_a, 1)} : 1)")
print(
    f"DeepSeek-V3.2: each query reads {fmt(ref['DeepSeek-V3.2']['topk'], 0)} selected tokens, but all "
    f"{fmt(T_LONG, 0)} are kept"
)

## 11 · Try changing things

The cells below rerun the same functions with different parameters. You can edit the values and run them again.

### The state size $d_k$ of the untrained memory
Section 7 found that trained memory grows with $d_k$. Does the untrained memory do the same? The cell below draws the
keys and value words again in $d_k$ dimensions (with $d_v = d_k$; the draws differ from the demo's, because the demo
uses only $d_k = 16$).

In [ ]:
for dk in (8, 16, 32):
    c_, k_, i_, v_ = memory_draws(draws=200, dk=dk)
    row = []
    for N in (8, 16, 32, 64):
        a_add = read_back(write("add", k_[:, :N], v_[:, :N]), c_, k_[:, :N], i_[:, :N]).mean()
        a_ls = read_back(write("ls", k_[:, :N], v_[:, :N]), c_, k_[:, :N], i_[:, :N]).mean()
        row.append(f"N {N}: {pct(a_add, 0)} / {pct(a_ls, 0)}")
    print(f"d_k = {dk:>2} (add / least squares): " + " · ".join(row))

Least squares always fits exactly up to $N = d_k$. With adding, the accuracy at $N = d_k$ is 55% ($d_k = 8$), 83% (16)
and 96% (32): in more dimensions, random keys overlap less (mean $\cos^2$ of $1/d_k$), so memory grows even faster
than $d_k$.

### A fixed forget gate: remember longer, or remember less?
A constant gate $\alpha$ trades remembering old pairs against reducing overlap. Try a few values with 32 untrained
pairs, split into 4 groups in writing order as in section 5:

In [ ]:
N = 32
for alpha in (0.8, 0.9, 0.95, 0.99, 1.0):
    h = read_back(write("gate", keys[:, :N], vals[:, :N], alpha=alpha), code, keys[:, :N], ids[:, :N])
    groups = " / ".join(pct(g.mean(), 0) for g in np.array_split(h, 4, axis=1))
    print(f"α = {fmt(alpha, 2)}: {pct(h.mean(), 1)} correct  (old → new: {groups})")

No value $\alpha < 1$ beats adding ($\alpha = 1$) on this task: forgetting only helps when there is something *worth*
forgetting, like the old segment in section 8. A constant gate can't tell the two cases apart; a per-token gate can.

### Training again with a different seed, or without a gate
Each run takes a few seconds. Change `seed` or `gate` (`"token"`, `"fixed"`, `"none"`) and compare with the results
file's 3-seed range.

In [ ]:
for gate, seed in (("token", 1), ("none", 0)):
    lr = REC["runs"]["reset-gated-N16-d16" if gate == "token" else "reset-add-N16-d16"]["lr"]
    p_ = train("reset", 16, gate, lr, seed=seed)
    acc, f_, _ = test_acc(p_, gate)
    ref_acc = accs("reset", "gated" if gate == "token" else "add", 16)
    print(
        f"gate {gate:<5} seed {seed} (η = {fmt(lr, 2)}): {pct(acc, 1)} correct, α at the separator "
        f"{fmt(np.exp(f_['la'][:, 16]).mean(), 3)}  ·  results file (6,000 steps, 3 seeds): {span(ref_acc)}"
    )

Trained for 800 steps, both are a little below the results file's 3-seed range, but the order doesn't change: with a
different seed, the per-token gate still finds the separator by itself, while the plain sum has no way to erase the
first group.

## 12 · Limits and next steps

- **A toy task, one layer, one head.** Real models stack dozens of layers, each with many heads, and add a short
  convolution before $q, k, v$ and a gate on the output. Key–value recall is only one aspect of language modeling,
  although an important one: Arora et al. ([2023](https://arxiv.org/abs/2312.04927)) found that it explains 82% of the
  perplexity gap between attention and gated convolutional models.
- **No speed measurements.** The notebook only counts memory. Linear layers are cheap when *generating* (each token
  updates a fixed-size state), and fast when *training* thanks to the chunked form, which runs as matrix
  multiplications on GPUs, with custom-written kernels.
- **A fixed memory is still a fixed memory.** No gate or write rule makes a $16 \times 16$ matrix hold 64 arbitrary
  pairs exactly in a single reading pass. That is exactly why hybrid models keep a few full attention layers.

Lessons 22–25 are about **base models**: pretraining, scaling laws, MoE, and now architectures for long texts. But a
base model only knows how to continue text. Lesson 26 turns it into an assistant: supervised learning on example
answers, then reinforcement learning with rewards that a program can check.

## Summary
- The KV cache grows linearly with length: Llama 3 8B needs 128 KiB per token, i.e. 32 GiB at 262,144 tokens; Llama 2
  7B, which has no GQA, needs 128 GiB, exactly the figure in Table 1 of the Jamba paper.
- Drop the softmax, and attention can be written as an RNN with the state $S_t = \alpha_t S_{t-1} + \beta_t k_t^\top v_t$,
  $o_t = q_t S_t$. The recurrent form, the masked matrix form and the chunked form are the same computation, with or
  without the delta rule: they differ by about $10^{-14}$ at 2,048 tokens.
- An untrained $16 \times 16$ matrix that stores 16 pairs with random keys by adding reads back only 83.1% correctly,
  because the keys overlap; with orthogonal keys, 100.0%. Least squares fits exactly up to 16 pairs, then only 79.0%
  at 32 pairs.
- The delta rule is one step of gradient descent on $\tfrac12\lVert k_t S - v_t\rVert^2$ with learning rate $\beta_t$;
  repeated with a small $\beta$, it moves toward the least-squares solution.
- After training, a $16 \times 16$ state: ≥ 99% up to 8 pairs, 79–80% at 32 and 48–49% at 64 (elu + 1: 66% and 37%;
  softmax: 100%). A $32 \times 32$ state brings 32 pairs up to 99.98–99.99%.
- A per-token forget gate learns by itself to erase the old segment: $\alpha = 0.0036$ at the separator (results
  file), 96.9–97.3% correct against 88.6–88.9% for a constant gate. When a key is written again, the delta rule is 91%
  correct, the forget gate only 78%.
- Mamba-2 is gated linear attention with $(C, B, x) \to (q, k, v)$, $\alpha_t = e^{-\Delta_t a}$, $\beta_t = \Delta_t$;
  take away “selectivity” and what is left is a convolution (S4).
- Hybrid models: Qwen3-Next interleaves 3 Gated DeltaNet layers with 1 attention layer, only 24 KiB per token. At
  262,144 tokens it needs 6.04 GiB, against 32 GiB for Llama 3 8B.

## Review questions
1. Why does Llama 2 7B need a KV cache 4 times as large as Llama 3 8B's, even though it has fewer parameters?
2. The recurrent form and the matrix form give the same $O$. Which form is used when generating text, which when
   training, and why does the RNN of Lesson 15 have no chunked form?
3. Writing 16 pairs with random keys into a $16 \times 16$ matrix by adding reads back only 83.1% correctly. Where do
   the errors come from, and when is it 100% correct?
4. The delta rule is one step of gradient descent on which loss? With $\beta = 1$ and $\lVert k \rVert = 1$, what does
   it do to the pair just written?
5. On the overwrite task, why is the forget gate only 78% correct while the delta rule is 91% correct? Why does softmax
   without positions stop at about 75%?
6. Where does Mamba-2 differ from gated linear attention? What makes Mamba different from S4?

<details><summary>Hints and answers</summary>

1. Llama 3 uses GQA: 8 KV heads for 32 query heads, so each layer keeps only 8 pairs of K, V per token instead of 32.
2. Generating text uses the recurrent form: each new token only updates and reads the fixed-size state. Training uses
   the chunked (or matrix) form: the whole sequence is already available, so it is computed in parallel with matrix
   multiplications. The RNN of Lesson 15 has a $\tanh$ inside the loop, so $h_t$ doesn't depend linearly on $h_{t-1}$,
   and several steps can't be merged into one matrix multiplication.
3. Reading back gives $o_i = v_i + \sum_{j \ne i} (k_i \cdot k_j) v_j$: random keys aren't orthogonal to one another, so
   the other pairs add noise. With $N \le d_k$ orthogonal keys, every product $k_i \cdot k_j$ ($i \ne j$) is 0 and the
   read-back is exactly right.
4. $\tfrac12\lVert k_t S - v_t\rVert^2$, with learning rate $\beta_t$. With $\beta = 1$, $k_t S_t = v_t$: the error on
   the pair just written drops to exactly 0, and the old value of $k_t$ is erased and replaced with the new one.
5. The forget gate shrinks the *whole* state, so it can't erase the old value of a single key; the delta rule subtracts
   exactly the part $k_t S_{t-1}$ of the key being written. Softmax without positions sees both the old and the new
   value and doesn't know which one came later: it gets every unchanged key right and about half of the rewritten
   keys.
6. Nowhere: Mamba-2 is gated linear attention, with only the renaming $(C, B, x) \to (q, k, v)$, with
   $\alpha_t = e^{-\Delta_t a}$ and $\beta_t = \Delta_t$. Mamba makes $\Delta, B, C$ depend on the token (“selective”);
   S4 uses fixed parameters, so the whole layer is a convolution.
</details>

## Further reading
- A. Katharopoulos et al. (2020), [Transformers are RNNs](https://arxiv.org/abs/2006.16236): linear attention with
  $\varphi = \operatorname{elu} + 1$.
- I. Schlag, K. Irie, J. Schmidhuber (2021), [Linear Transformers Are Secretly Fast Weight Programmers](https://arxiv.org/abs/2102.11174):
  the memory limit and the delta rule.
- S. Yang et al. (2024), [Parallelizing Linear Transformers with the Delta Rule over Sequence Length](https://arxiv.org/abs/2406.06484),
  and the blog [DeltaNet Explained](https://sustcsonglin.github.io/blog/2024/deltanet-1/): the delta rule as online
  regression, the UT form.
- S. Yang, J. Kautz, A. Hatamizadeh (2024), [Gated Delta Networks](https://arxiv.org/abs/2412.06464).
- A. Gu, T. Dao (2023), [Mamba](https://arxiv.org/abs/2312.00752); T. Dao, A. Gu (2024), [Mamba-2 / SSD](https://arxiv.org/abs/2405.21060)
  and [the blog, parts I–IV](https://tridao.me/blog/2024/mamba2-part1-model/).
- S. Arora et al. (2023), [Zoology](https://arxiv.org/abs/2312.04927) and (2024), [Based](https://arxiv.org/abs/2402.18668):
  key–value recall and the tradeoff between state size and memory.
- S. Jelassi et al. (2024), [Repeat After Me](https://arxiv.org/abs/2402.01032): Transformers copy strings better than
  models with a fixed-size state.
- Y. Sun et al. (2024), [Learning to (Learn at Test Time)](https://arxiv.org/abs/2407.04620): the state is a model that
  is learned on the fly, while reading.
- L. Welch (1974), [Lower bounds on the maximum cross correlation of signals](https://doi.org/10.1109/TIT.1974.1055219).
- Hybrid models and other directions: [Jamba](https://arxiv.org/abs/2403.19887), [MiniMax-01](https://arxiv.org/abs/2501.08313),
  [Qwen3-Next](https://huggingface.co/Qwen/Qwen3-Next-80B-A3B-Instruct), [Kimi Linear](https://arxiv.org/abs/2510.26692),
  [DeepSeek-V3.2](https://huggingface.co/deepseek-ai/DeepSeek-V3.2) and
  [Why Did MiniMax M2 End Up as a Full Attention Model?](https://huggingface.co/blog/MiniMax-AI/why-did-m2-end-up-as-a-full-attention-model).

**Next lesson:** [Lesson 26 · From language model to assistant](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/hau-huan-luyen/):
turning a model that only knows how to continue text into an assistant that answers questions correctly.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")